In [14]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


In [15]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

## Online MCP

In [16]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pprint import pprint


In [17]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Tehran"
            ]
        }
    }
)
tools = await client.get_tools()

In [20]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [21]:
question = HumanMessage(content="الان ساعت چنده؟")

response = await agent.ainvoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

الان ساعت 17:56:09 شنبه 10 اکتبر 2026 است. منطقه زمانی: Asia/Tehran (IRST, UTC+3:30).


In [25]:
question = HumanMessage(content="الان ساعت در برلین چنده؟")

response = await agent.ainvoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

الان در برلین ساعت 16:29:10 است (CEST، UTC+2). شنبه ۱۰ اکتبر ۲۰۲۶. اگر به زمان منطقهٔ دیگری هم نیاز دارید بگویید.


## Adding Filesystem MCP server

In [32]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Tehran"
            ]
        },
        "filesystem": {
            "transport": "stdio",
              "command": "npx",
              "args": [
                "-y",
                "@modelcontextprotocol/server-filesystem",
                "D:/Desktop/mcp-test",
              ]
            }
    }
)
tools = await client.get_tools()

In [34]:
for tool in tools:
    print(tool.name)

get_current_time
convert_time
read_file
read_text_file
read_media_file
read_multiple_files
write_file
edit_file
create_directory
list_directory
list_directory_with_sizes
directory_tree
move_file
search_files
get_file_info
list_allowed_directories


In [35]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [36]:
question = HumanMessage(content="چه فایلهایی دارم؟")

response = await agent.ainvoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

گزارش فایل‌های موجود در دایرکتوری D:\Desktop\mcp-test:

- salaam.txt
  - اندازه: 3 بایت
  - ایجاد شده: Sat Oct 10 2026 18:14:23 GMT+0330 (Iran Standard Time)
  - ویرایش شده: Sat Oct 10 2026 18:14:29 GMT+0330 (Iran Standard Time)
  - دسترسی: 18:14:30 (Last accessed)
  - نوع: فایل
  - مجوزها: 666 (خواندنی/نوشتنی برای همه کاربران)
  - محتوا: bye

اگر می‌خواهی فایلی جدید بسازی یا محتوای این فایل را تغییر بدهیم یا بخوانیم، بگو تا انجام بدهم.


In [37]:
question = HumanMessage(content="ساعت و تاریخ الان را در فایل now.txt بنویس")

response = await agent.ainvoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

تایم‌استمپ فعلی در ایران (Asia/Tehran) نوشته شد:
2026-10-10T18:18:34+03:30

اگر می‌خواهید فرمت دیگری از تاریخ و زمان یا تنها زمان (مثلاً فقط ساعت) نوشته شود، بگویید تا تغییرش بدهم.


More MCP servers:

https://github.com/punkpeye/awesome-mcp-servers

https://mcp.so/servers?tag=featured

